In [2]:
from keras.models import load_model
import numpy as np

# Load the model
model = load_model('ASL.h5')

# Check model summary and input shape
print("Model Summary:")
model.summary()

# Get input shape
input_shape = model.input_shape
print(f"\nExpected input shape: {input_shape}")

# Check if it's a list (for functional API) or tuple (for Sequential)
if isinstance(input_shape, list):
    input_shape = input_shape[0]
    print(f"First input shape: {input_shape}")

# Print the actual shape dimensions
height, width, channels = input_shape[1], input_shape[2], input_shape[3]
print(f"Expected: Height={height}, Width={width}, Channels={channels}")

Model Summary:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 60, 60, 32)     │         2,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 60, 60, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 28, 28, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 28, 28, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 29)             │         3,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 356,639 (1.36 MB)

 Trainable params: 356,637 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)


Expected input shape: (None, 64, 64, 3)
Expected: Height=64, Width=64, Channels=3


In [3]:
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
from keras.preprocessing import image
import numpy as np

def predict_single_image_fixed(model, image_path):
    """
    Fixed prediction function with debugging
    """
    # Get model input shape
    input_shape = model.input_shape
    if isinstance(input_shape, list):
        input_shape = input_shape[0]
    
    height, width, channels = input_shape[1], input_shape[2], input_shape[3]
    print(f"Model expects input shape: (batch_size, {height}, {width}, {channels})")
    
    try:
        # Load and preprocess the image
        img = image.load_img(image_path, target_size=(height, width))
        img_array = image.img_to_array(img)
        
        print(f"Original image shape: {img_array.shape}")
        
        # Ensure we have the right number of channels
        if len(img_array.shape) == 3 and img_array.shape[2] == 3:
            # RGB image - good
            pass
        elif len(img_array.shape) == 2:
            # Grayscale - convert to RGB
            img_array = np.stack((img_array,)*3, axis=-1)
            print(f"Converted grayscale to RGB: {img_array.shape}")
        else:
            # Wrong number of channels
            if channels == 1:
                # Model expects grayscale
                if len(img_array.shape) == 3:
                    img_array = np.mean(img_array, axis=-1, keepdims=True)
                print(f"Converted to grayscale: {img_array.shape}")
            else:
                # Model expects RGB
                if len(img_array.shape) == 3 and img_array.shape[2] == 1:
                    img_array = np.repeat(img_array, 3, axis=-1)
                print(f"Ensured RGB format: {img_array.shape}")
        
        # Add batch dimension
        img_array = np.expand_dims(img_array, axis=0)
        print(f"After expand_dims: {img_array.shape}")
        
        # Normalize (try both [0,1] and [-1,1] to see what works)
        img_array_normalized = img_array / 255.0  # [0,1] range
        print(f"After normalization [0,1]: {img_array_normalized.shape}")
        print(f"Data range: min={img_array_normalized.min():.3f}, max={img_array_normalized.max():.3f}")
        
        # Make prediction
        print("Making prediction...")
        prediction = model.predict(img_array_normalized, verbose=0)
        print(f"Prediction shape: {prediction.shape}")
        
        predicted_class = np.argmax(prediction[0])
        confidence = prediction[0][predicted_class]
        
        return predicted_class, confidence, prediction
        
    except Exception as e:
        print(f"Error during prediction: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None

# Usage
image_path = 'test_image.jpg'

# First, check model info
print("=== MODEL INFORMATION ===")
model = load_model('ASL.h5')
input_shape = model.input_shape
if isinstance(input_shape, list):
    input_shape = input_shape[0]
print(f"Model input shape: {input_shape}")

# Try prediction
print("\n=== PREDICTION ATTEMPT ===")
predicted_class, confidence, full_prediction = predict_single_image_fixed(model, image_path)

if predicted_class is not None:
    print(f"\n=== RESULTS ===")
    print(f"Predicted class: {predicted_class}")
    print(f"Confidence: {confidence:.2f}")
    
    # Show top 3 predictions
    top_3_indices = np.argsort(full_prediction[0])[-3:][::-1]
    print("\nTop 3 predictions:")
    for i, idx in enumerate(top_3_indices):
        prob = full_prediction[0][idx]
        print(f"  {i+1}. Class {idx}: {prob:.3f}")
else:
    print("Prediction failed!")

=== MODEL INFORMATION ===


Model input shape: (None, 64, 64, 3)

=== PREDICTION ATTEMPT ===
Model expects input shape: (batch_size, 64, 64, 3)
Original image shape: (64, 64, 3)
After expand_dims: (1, 64, 64, 3)
After normalization [0,1]: (1, 64, 64, 3)
Data range: min=0.000, max=1.000
Making prediction...
Prediction shape: (1, 29)

=== RESULTS ===
Predicted class: 0
Confidence: 1.00

Top 3 predictions:
  1. Class 0: 1.000
  2. Class 18: 0.000
  3. Class 4: 0.000


In [4]:
def predict_with_different_normalization(model, image_path):
    """
    Try different normalization methods
    """
    input_shape = model.input_shape
    if isinstance(input_shape, list):
        input_shape = input_shape[0]
    
    height, width, channels = input_shape[1], input_shape[2], input_shape[3]
    
    # Load image
    img = image.load_img(image_path, target_size=(height, width))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    
    # Try different preprocessing methods
    preprocessing_methods = [
        ("No normalization", img_array),
        ("Divide by 255 [0,1]", img_array / 255.0),
        ("StandardScaler-like [-1,1]", (img_array / 127.5) - 1.0),
        ("Divide by 255 and subtract 0.5", (img_array / 255.0) - 0.5),
    ]
    
    for name, processed_img in preprocessing_methods:
        try:
            print(f"\nTrying: {name}")
            print(f"Input shape: {processed_img.shape}")
            print(f"Data range: [{processed_img.min():.3f}, {processed_img.max():.3f}]")
            
            prediction = model.predict(processed_img, verbose=0)
            print(f"✓ Success! Output shape: {prediction.shape}")
            
            predicted_class = np.argmax(prediction[0])
            confidence = prediction[0][predicted_class]
            print(f"Predicted: Class {predicted_class} ({confidence:.3f})")
            return predicted_class, confidence
            
        except Exception as e:
            print(f"✗ Failed: {e}")
            continue
    
    return None, None

In [2]:
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
from keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt

def safe_predict(model, image_path):
    """
    Safe prediction with automatic shape detection and error handling
    """
    try:
        # Get model requirements
        input_shape = model.input_shape
        if isinstance(input_shape, list):
            input_shape = input_shape[0]
        
        height, width, channels = input_shape[1], input_shape[2], input_shape[3]
        print(f"Model expects: {height}x{width}x{channels}")
        
        # Load image
        img = image.load_img(image_path, target_size=(height, width))
        img_array = image.img_to_array(img)
        print(f"Loaded image shape: {img_array.shape}")
        
        # Handle channel conversion
        if channels == 1 and img_array.shape[-1] == 3:
            # Convert RGB to grayscale
            img_array = np.dot(img_array[...,:3], [0.2989, 0.5870, 0.1140])
            img_array = np.expand_dims(img_array, axis=-1)
        elif channels == 3 and img_array.shape[-1] == 1:
            # Convert grayscale to RGB
            img_array = np.repeat(img_array, 3, axis=-1)
        
        # Add batch dimension
        img_array = np.expand_dims(img_array, axis=0)
        print(f"Final input shape: {img_array.shape}")
        
        # Normalize - try the most common methods
        img_array_test = img_array.astype('float32')
        
        # Method 1: [0,1] range
        img_array_test1 = img_array_test / 255.0
        try:
            prediction = model.predict(img_array_test1, verbose=0)
            print("✓ [0,1] normalization worked!")
            return prediction
        except:
            pass
        
        # Method 2: No normalization
        try:
            prediction = model.predict(img_array_test, verbose=0)
            print("✓ No normalization worked!")
            return prediction
        except:
            pass
        
        # Method 3: [-1,1] range
        img_array_test3 = (img_array_test / 127.5) - 1.0
        try:
            prediction = model.predict(img_array_test3, verbose=0)
            print("✓ [-1,1] normalization worked!")
            return prediction
        except:
            pass
        
        raise ValueError("None of the normalization methods worked!")
        
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        return None

# Main execution
if __name__ == "__main__":
    # Load model
    model = load_model('my_model.keras')
    
    # Check model info
    print("Model Input Shape:", model.input_shape)
    print("\nModel Summary:")
    model.summary()
    
    # Try prediction
    image_path = 'D_test.jpg'  # Make sure this file exists!
    
    # Verify file exists
    import os
    if not os.path.exists(image_path):
        print(f"Error: {image_path} does not exist!")
        print("Please make sure you have a test image in the current directory.")
        print("You can use any .jpg, .png, or .jpeg file.")
    else:
        prediction = safe_predict(model, image_path)
        
        if prediction is not None:
            predicted_class = np.argmax(prediction[0])
            confidence = prediction[0][predicted_class]
            
            print(f"\n=== FINAL RESULT ===")
            print(f"Predicted Class: {predicted_class}")
            print(f"Confidence: {confidence:.2f}")
            
            # Show top 5 predictions
            top_indices = np.argsort(prediction[0])[-5:][::-1]
            print("\nTop 5 Predictions:")
            for i, idx in enumerate(top_indices):
                prob = prediction[0][idx]
                print(f"{i+1}. Class {idx}: {prob:.3f} ({prob*100:.1f}%)")

Model Input Shape: (None, 224, 224, 3)

Model Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 29)             │         7,453 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,791,740 (18.28 MB)

 Trainable params: 2,197,341 (8.38 MB)

 Non-trainable params: 397,056 (1.51 MB)

 Optimizer params: 2,197,343 (8.38 MB)

Model expects: 224x224x3
Loaded image shape: (224, 224, 3)
Final input shape: (1, 224, 224, 3)
✓ [0,1] normalization worked!

=== FINAL RESULT ===
Predicted Class: 3
Confidence: 1.00

Top 5 Predictions:
1. Class 3: 1.000 (100.0%)
2. Class 8: 0.000 (0.0%)
3. Class 23: 0.000 (0.0%)
4. Class 11: 0.000 (0.0%)
5. Class 5: 0.000 (0.0%)


In [ ]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.preprocessing import image

# Path to your test folder
test_dir = r"asl_alphabet_test\asl_alphabet_test"

# Load your trained model (update with your saved model path if needed)
model = tf.keras.models.load_model("my_model.keras")

# Make predictions for each image in the test folder
for filename in os.listdir(test_dir):
    if filename.endswith(".jpg") or filename.endswith(".png"):
        img_path = os.path.join(test_dir, filename)

        # Load image and preprocess (resize to match your training input size, e.g., 224x224)
        img = image.load_img(img_path, target_size=(224, 224))
        img_array = image.img_to_array(img) / 255.0  # normalize
        img_array = np.expand_dims(img_array, axis=0)  # add batch dimension

        # Predict
        prediction = model.predict(img_array, verbose=0)
        predicted_class = np.argmax(prediction, axis=1)[0]

        print(f"{filename} --> Predicted Class: {predicted_class}")


A_test.jpg --> Predicted Class: 0
B_test.jpg --> Predicted Class: 1
C_test.jpg --> Predicted Class: 2
D_test.jpg --> Predicted Class: 3
E_test.jpg --> Predicted Class: 4
F_test.jpg --> Predicted Class: 5
G_test.jpg --> Predicted Class: 6
H_test.jpg --> Predicted Class: 7
I_test.jpg --> Predicted Class: 8
J_test.jpg --> Predicted Class: 9
K_test.jpg --> Predicted Class: 10
L_test.jpg --> Predicted Class: 11
M_test.jpg --> Predicted Class: 12
nothing_test.jpg --> Predicted Class: 27
N_test.jpg --> Predicted Class: 13
O_test.jpg --> Predicted Class: 14
P_test.jpg --> Predicted Class: 15
Q_test.jpg --> Predicted Class: 16
R_test.jpg --> Predicted Class: 17
space_test.jpg --> Predicted Class: 28
S_test.jpg --> Predicted Class: 18
T_test.jpg --> Predicted Class: 19
U_test.jpg --> Predicted Class: 20
V_test.jpg --> Predicted Class: 21
W_test.jpg --> Predicted Class: 22
X_test.jpg --> Predicted Class: 23
Y_test.jpg --> Predicted Class: 24
Z_test.jpg --> Predicted Class: 25


: 